# Lecture 4: Pontryagin's minimum principle

**The challenge.** Every condition so far came from $H_u = 0$: perturb the
control a little in any direction, and the cost must not change to first
order. That argument fails the moment the control is *limited*. A motor
delivers at most $u_{\max}$; a valve is open or shut; a rocket engine points
in a direction, a unit vector, not in an arbitrary vector. At a bound you
cannot perturb in both directions, so $H_u$ need not vanish there. Worse, for
a minimum-time problem $H$ is *linear* in the control: $H_u = 0$ has no
solution at all, and the calculus of variations of Lectures 2 and 3 has
nothing to say.

The answer, found in Moscow in 1956, is that the optimal control does not
make $H$ stationary: it makes $H$ **as small as possible** over the
admissible controls, instant by instant. This lecture states that principle,
sketches why it holds, and works out what it says for every control set the
package supports.

**After this lecture you can**

- state the minimum principle and explain the needle variation behind it;
- derive the optimal control law for unbounded, clipped, bang-bang and
  unit-vector controls;
- read the law, and the switching function, from what `derive` returns.

## 1. A blind topologist turns to engineering

Lev Semyonovich Pontryagin (1908–1988) lost his sight at fourteen, when a
primus stove exploded in his face. His mother, Tatyana Andreevna, a
seamstress with no mathematical training, became his reader. She read him
textbooks and papers, including ones in German that she could not
understand, describing each formula symbol by symbol, inventing her own names
for the ones she did not know. He went to Moscow University, and by his
thirties he was one of the leading topologists in the world: Pontryagin
duality and the Pontryagin classes are named after him.

Around 1952 he decided, by his own account, that a mathematician should do
something useful, and he began a seminar at the Steklov Institute with three
young colleagues: Vladimir Boltyanskii, Revaz Gamkrelidze and Evgenii
Mishchenko. They went looking for problems among engineers, including
officers of the Soviet Air Force, who brought them questions like how to
bring an aircraft to a given altitude and speed in the least time. The
controls in those problems were bounded, and the classical calculus of
variations could not handle them. In 1956 the group announced a **maximum
principle** in the *Doklady* of the Academy of Sciences; Boltyanskii found a
complete proof in 1958; the book, *The Mathematical Theory of Optimal
Processes*, appeared in 1961 and in English in 1962, the year the four
received the Lenin Prize.

They were not alone. At the RAND Corporation in Santa Monica, Magnus Hestenes
had written a 1950 research memorandum on minimum-time aircraft climbs that
contains an equivalent condition, phrased in the language of the calculus of
variations. It circulated mostly inside RAND. The Moscow group's statement,
general and complete with a proof, is the one the field adopted.

Pontryagin's sign convention maximizes a function equal to $-H$. Most
engineering texts, and this package, *minimize* $H$, so we call it the
minimum principle. They are the same theorem.

## 2. The principle

> **Pontryagin's minimum principle.** If $u^*$ is optimal, with state $x^*$,
> there is a costate $\lambda(t)$ such that
>
> $$
> \dot x^* = H_\lambda^\top, \qquad \dot\lambda = -H_x^\top, \qquad
> u^*(t) = \arg\min_{v \in U} H\big(t, x^*(t), \lambda(t), v\big),
> $$
>
> with the transversality conditions of Lecture 2, and, when the final time
> is free, $H(t_f) = 0$ (for an autonomous problem, $H \equiv 0$).

Strictly, $H = \lambda_0 L + \lambda^\top f$ with a constant $\lambda_0 \ge 0$
that is not identically zero together with $\lambda$. The **abnormal** case
$\lambda_0 = 0$ (the cost plays no part) occurs only in degenerate problems,
and the package assumes the normal case $\lambda_0 = 1$.

**Why minimize rather than differentiate.** Pontryagin's proof uses a
*needle variation*: replace the optimal control by an arbitrary admissible
value $v$ on a tiny interval $[\tau, \tau + \varepsilon]$, and leave it alone
everywhere else. The change can be large in size but is short in time, so the
state changes by $O(\varepsilon)$. That perturbation then propagates by the
linearized dynamics to the end, and the costate is exactly what converts it
into a change of cost. To first order,
$$
\Delta J = \varepsilon \,\Big[ H\big(\tau, x^*, \lambda, v\big) - H\big(\tau, x^*, \lambda, u^*\big) \Big] .
$$
Optimality needs $\Delta J \ge 0$ for every $v \in U$ and almost every
$\tau$, which is the minimum condition. Nothing in the argument needs $v$ to
be close to $u^*$, or $U$ to be anything but a set. (Making the argument
rigorous, in particular for terminal constraints, is where the work went.)
Where the minimizer is interior and $H$ is smooth, the minimum condition
reduces to $H_u = 0$, so the earlier lectures are a special case.

In [ ]:
# A needle variation on a double integrator x'' = u: the control is replaced by an
# arbitrary admissible value v on [tau, tau + eps]. The state moves by O(eps), however far
# v is from u*.
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
t = np.linspace(0, 1, 2001)
u_star = 1.0 - 2.0 * t
tau, eps, v_needle = 0.3, 0.015, -1.5


def simulate(u):
    dt = t[1] - t[0]
    vel = np.concatenate([[0.0], np.cumsum((u[1:] + u[:-1]) / 2) * dt])
    pos = np.concatenate([[0.0], np.cumsum((vel[1:] + vel[:-1]) / 2) * dt])
    return pos, vel


u_needle = np.where((t >= tau) & (t < tau + eps), v_needle, u_star)
x0, v0 = simulate(u_star)
x1, v1 = simulate(u_needle)

fig, (left, mid, right) = plt.subplots(1, 3, figsize=(12.5, 3.4))
left.plot(t, u_star, color=BLUE, lw=2, label="$u^*(t)$")
left.plot(t, u_needle, color=ORANGE, lw=1.5, label="with a needle")
left.axhspan(-1.6, 1.6, color=GRAY, alpha=0.08, lw=0)
left.text(0.02, 1.45, "U = [-1.6, 1.6]", fontsize=8)
left.annotate(r"value $v \in U$ on $[\tau, \tau + \varepsilon]$", xy=(tau + eps / 2, v_needle),
              xytext=(0.42, -1.3), fontsize=9, arrowprops={"arrowstyle": "-|>", "lw": 0.8})
left.set(xlabel="t", ylabel="u", ylim=(-1.7, 1.7))
left.set_title("the control: a large change for a short time", fontsize=10, loc="left")
left.legend(frameon=False, fontsize=8, loc="upper right")

mid.plot(t, x0, color=BLUE, lw=2, label="x, optimal")
mid.plot(t, x1, color=ORANGE, lw=1.5, ls="--", label="x, with the needle")
mid.set(xlabel="t", ylabel="position x")
mid.set_title(r"the state: moved by $O(\varepsilon)$", fontsize=10, loc="left")
mid.legend(frameon=False, fontsize=8)

right.plot(t, (v1 - v0) / eps, color=ORANGE, lw=2, label=r"$\delta v / \varepsilon$")
right.plot(t, (x1 - x0) / eps, color=BLUE, lw=2, label=r"$\delta x / \varepsilon$")
right.axvline(tau, color="k", lw=0.8, ls=":")
right.set(xlabel="t", ylabel=r"perturbation / $\varepsilon$")
right.set_title(r"the change, divided by $\varepsilon$, carried to $t_f$", fontsize=10,
                loc="left")
right.legend(frameon=False, fontsize=8)
for a in (left, mid, right):
    a.grid(alpha=0.25)
fig.tight_layout()

*A needle variation on $\ddot x = u$. The control jumps to a far-away admissible value for a short time $\varepsilon$. The state hardly moves, and the change it does make, divided by $\varepsilon$, is a fixed perturbation carried forward by the linearized dynamics. That is why the first-order change in cost is $\varepsilon\,[H(v) - H(u^*)]$ even though $v$ is not close to $u^*$.*

## 3. What the principle says for each control set

The minimization is over the control alone, at fixed $(t, x, \lambda)$. It
is a small finite-dimensional problem, and for the control sets that occur in
practice it has a closed form. Write $H = H_0 + b^\top u + \tfrac12 u^\top Q u$
in the controls.

| control set | $H$ in $u$ | optimal law | name |
|---|---|---|---|
| none | convex quadratic | $u = -Q^{-1} b$ ($H_u = 0$) | |
| `Box(u, lo, hi)` | convex quadratic | $u = \operatorname{clip}(-b/Q, \text{lo}, \text{hi})$ | **saturated** |
| `Box(u, lo, hi)` | linear, $H_0 + S\,u$ | $u = \text{hi}$ if $S < 0$, $\text{lo}$ if $S > 0$ | **bang-bang**; $S$ is the **switching function** |
| `UnitVector(u)` | linear, $b^\top u$ | $u = -b / \lVert b \rVert$ | pointing; for thrust, the **primer vector** |

Two things can go wrong. If $S \equiv 0$ on an interval, the bang-bang law
says nothing; that is a **singular arc**, Lecture 12. If $b = 0$, the
pointing direction is undefined. Both are special; the package detects the
first and assumes away the second.

`derive` decides which case applies by inspecting $H$. It proves convexity
of a quadratic $H$ from the declared signs of the symbols, which is one
reason to declare masses and gains `positive=True`. Here are the four cases on
the package's examples.

In [ ]:
# The minimum principle for each control set: H as a function of u at one instant,
# and where its minimum over U lies.
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
fig, axes = plt.subplots(1, 4, figsize=(13, 3.3))
u = np.linspace(-2, 2, 400)
lo, hi = -1.0, 1.0


def mark(ax, x, y, text, dx=0.1, dy=0.3):
    ax.plot(x, y, "o", color=ORANGE, ms=9, zorder=5)
    ax.annotate(text, (x, y), xytext=(x + dx, y + dy), fontsize=9)


ax = axes[0]
H = 0.5 * (u - 0.6) ** 2
ax.plot(u, H, color=BLUE, lw=2)
mark(ax, 0.6, 0, r"$H_u = 0$", -0.3, 0.4)
ax.set_title("no set: quadratic H", fontsize=10, loc="left")

ax = axes[1]
H = 0.5 * (u - 1.6) ** 2
inside = (u >= lo) & (u <= hi)
ax.plot(u, H, color=BLUE, lw=1, alpha=0.4)
ax.plot(u[inside], H[inside], color=BLUE, lw=2.5)
ax.plot(1.6, 0, "o", color=GRAY, ms=6)
ax.annotate("unconstrained\nminimum", (1.6, 0), xytext=(0.95, 1.6), fontsize=8, color="k",
            arrowprops={"arrowstyle": "-", "lw": 0.6, "color": GRAY})
mark(ax, hi, 0.5 * (hi - 1.6) ** 2, "clipped", -1.3, 0.2)
ax.set_title("Box, quadratic H: saturated", fontsize=10, loc="left")

ax = axes[2]
for S, color, label in [(-0.8, BLUE, "S < 0: u = hi"), (0.8, GRAY, "S > 0: u = lo")]:
    H = 1 + S * u
    ax.plot(u[inside], H[inside], color=color, lw=2.5, label=label)
    end = hi if S < 0 else lo
    ax.plot(end, 1 + S * end, "o", color=ORANGE, ms=9, zorder=5)
ax.legend(frameon=False, fontsize=8, loc="upper center")
ax.set_ylim(0, 2.4)
ax.set_title(r"Box, linear H $= H_0 + S u$: bang-bang", fontsize=10, loc="left")

for a, y_text in zip(axes[:3], [None, 0.92, 0.05]):
    if y_text is not None:
        a.axvspan(lo, hi, color=GRAY, alpha=0.1, lw=0)
        a.text(0, a.get_ylim()[1] * y_text, "U", ha="center", fontsize=10, color="k")
    a.set(xlabel="u", ylabel="H")
    a.grid(alpha=0.25)

ax = axes[3]
th = np.linspace(0, 2 * np.pi, 400)
b = np.array([0.8, 0.5])
H = b[0] * np.cos(th) + b[1] * np.sin(th)
ax.scatter(np.cos(th), np.sin(th), c=H, cmap="Blues", s=12)
ax.annotate("", xy=b, xytext=(0, 0), arrowprops={"arrowstyle": "-|>", "color": "k", "lw": 1.5})
ax.text(b[0] + 0.05, b[1] + 0.05, "b", fontsize=11, style="italic")
umin = -b / np.linalg.norm(b)
ax.annotate("", xy=umin, xytext=(0, 0), arrowprops={"arrowstyle": "-|>", "color": ORANGE, "lw": 2})
ax.plot(*umin, "o", color=ORANGE, ms=9)
ax.text(umin[0] - 0.1, umin[1] - 0.35, r"$u = -b/\|b\|$", fontsize=9, ha="center")
ax.text(0.0, 1.25, r"H $= b^\top u$ on the circle (darker = larger)", fontsize=8, ha="center")
ax.set(xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("UnitVector: point against b", fontsize=10, loc="left")
fig.tight_layout()

*The minimization in pictures, at one instant. A convex quadratic $H$ has its minimum where $H_u = 0$, or on the nearest bound if that lies outside the box. A linear $H$ is smallest at one end of the box, chosen by the sign of its slope $S$. On the unit circle, $b^\top u$ is smallest pointing against $b$.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control import examples
from auto_optimal_control.pontryagin import derive

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

for name in ["min_energy", "min_energy_bounded", "min_time", "zermelo"]:
    pmp = derive(examples.all_examples()[name])
    print(f"{name}:")
    for u, law in pmp.control_law.items():
        print(f"    {u} = {law}")

The unbounded law is linear in the costate; bounding it wraps it in
`Min`/`Max`, a clip; making the cost minimum time turns it into a
`Piecewise` on the sign of $\lambda_v$, the switching function; and Zermelo's
heading is the unit vector against $\lambda$.

For a bang-bang control, `derive` records the switching function and both
bounds in `switching_functions`. The law is "upper where `expr < 0`":

In [ ]:
sw = derive(examples.double_integrator_min_time()).switching_functions[0]
print("control:", sw.control, " S =", sw.expr, " lower:", sw.lower, " upper:", sw.upper)

### Checking the minimization

The law should minimize $H$ over the admissible set at *every* point, not
just along extremals. That is easy to test numerically: evaluate the law at
random $(x, \lambda)$ and compare $H$ there with $H$ on a dense grid of
admissible controls.

In [ ]:
def check_minimizes(name, n_points=200, seed=0):
    pmp = derive(examples.all_examples()[name])
    problem = pmp.problem
    rng = np.random.default_rng(seed)
    syms = [*pmp.states, *pmp.costates, *pmp.params]
    H = sp.lambdify([*syms, *pmp.controls], pmp.hamiltonian_full)
    law = sp.lambdify(syms, list(pmp.control_law.values()))
    box = {cs.control: cs for cs in problem.control_sets if hasattr(cs, "control")}
    worst = 0.0
    for _ in range(n_points):
        n_signed = len(pmp.states) + len(pmp.costates)     # parameters stay positive
        point = rng.uniform(0.2, 2.0, len(syms))
        point[:n_signed] *= rng.choice([-1, 1], n_signed)
        values = dict(zip(syms, point))
        best = np.array(law(*point), dtype=float)
        if len(pmp.controls) == 1:
            cs = box.get(pmp.controls[0])
            lo, hi = ((float(cs.lower.subs(values)), float(cs.upper.subs(values)))
                      if cs else (best[0] - 5, best[0] + 5))
            grid = np.linspace(lo, hi, 2001)[:, None]
        else:                      # a unit vector in the plane
            th = np.linspace(0, 2 * np.pi, 4001)
            grid = np.column_stack([np.cos(th), np.sin(th)])
        H_grid = np.array([H(*point, *g) for g in grid])
        worst = max(worst, H(*point, *best) - H_grid.min())
    print(f"{name:20s} max of H(u*) - min over the grid: {worst:.1e}")


for name in ["min_energy", "min_energy_bounded", "min_time", "zermelo"]:
    check_minimizes(name)

Never positive beyond rounding: the law is the minimizer. (The package's own
test suite runs this check on every example.)

## 4. Bounds change the answer, not just the control

It is tempting to think that a bounded problem is solved by taking the
unbounded solution and clipping it. It is not. Take the double integrator
$\ddot x = u$, from rest at $x = 1$ to rest at the origin in $T = 2$, with
cost $\int \tfrac12 u^2$. Unbounded, the optimal force is linear in time and
peaks at $|u| = 6/T^2 = 1.5$. Now allow only $|u| \le 1.2$.

In [ ]:
from auto_optimal_control.homotopy import Homotopy

T, a, b, umax = 2.0, 1.0, 0.0, 1.2
free = Homotopy(examples.double_integrator_min_energy()).solve([T, a, b], ([-5, -5], [5, 5]))
bounded = Homotopy(examples.double_integrator_min_energy(bounded=True)).solve(
    [T, a, b, umax], ([-5, -5], [5, 5]))
tr_free = free.shooting.trajectory(free.solution.v, n_points=401)
tr_bnd = bounded.shooting.trajectory(bounded.solution.v, n_points=401)

# The unbounded control, clipped, applied open loop (Euler steps are plenty here).
t_grid, u_clip = tr_free["t"], np.clip(tr_free["u"][:, 0], -umax, umax)
x_sim, v_sim = a, b
for k in range(len(t_grid) - 1):
    dt = t_grid[k + 1] - t_grid[k]
    x_sim, v_sim = x_sim + v_sim * dt + 0.5 * u_clip[k] * dt**2, v_sim + u_clip[k] * dt
print(f"unbounded optimum:  J = {free.objective:.6f}, max|u| = {np.abs(tr_free['u']).max():.3f}")
print(f"bounded optimum:    J = {bounded.objective:.6f}, max|u| = {np.abs(tr_bnd['u']).max():.3f}")
print(f"clipped unbounded:  ends at x = {x_sim:.4f}, v = {v_sim:.4f} (should be 0, 0)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(tr_free["t"], tr_free["u"][:, 0], color=GRAY, lw=2, label="unbounded optimum")
ax.plot(tr_free["t"], u_clip, color=ORANGE, lw=1.5, ls="--", label="unbounded, clipped")
ax.plot(tr_bnd["t"], tr_bnd["u"][:, 0], color=BLUE, lw=2, label="bounded optimum")
ax.axhline(umax, color="k", lw=0.8)
ax.axhline(-umax, color="k", lw=0.8)
ax.set(xlabel="t", ylabel="u")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()

Clipping the unbounded force loses impulse, so the mass never arrives. The
true bounded optimum holds the bound for longer and ramps through the middle
more steeply. It still follows $u = \operatorname{clip}(-\lambda_v)$, but with
a different costate: the bound changes $\lambda$, not just $u$. The kinks
where the law meets the bound are corners of the dynamics; Lecture 7 shows
how they are integrated.

## 5. Bang-bang control: the sign of the switching function

With a cost linear in the control, as in minimum time, the law is all or
nothing. For the double integrator from rest at $x = a$ to the origin, the
answer is full braking towards the origin, then full force the other way,
switching at $t = \sqrt{a / u_{\max}}$ and arriving at
$t_f = 2\sqrt{a/u_{\max}}$. Along the extremal, the control is decided by the
sign of $S = \lambda_v$:

In [ ]:
mt = Homotopy(examples.double_integrator_min_time()).solve(
    [1.0, 1.0, 0.0], ([-3, -3], [3, 3]), final_time=(0.5, 5.0))
tr = mt.shooting.trajectory(mt.solution.v, n_points=401)
print(f"t_f = {mt.solution.tf:.12f}  (closed form {2 * np.sqrt(1.0):.12f})")

fig, (top, bottom) = plt.subplots(2, 1, figsize=(7, 4.2), sharex=True)
top.plot(tr["t"], tr["z"][:, 3], color=BLUE, lw=2)
top.axhline(0, color="k", lw=0.8)
top.set_ylabel(r"$S = \lambda_v$")
bottom.plot(tr["t"], tr["u"][:, 0], color=ORANGE, lw=2)
bottom.set(xlabel="t", ylabel="u")
for a_ in (top, bottom):
    a_.grid(alpha=0.3)
fig.tight_layout()

$S$ is linear in time here ($\dot\lambda_v = -\lambda_x$, a constant), so it
crosses zero exactly once and the control switches exactly once. That is not
an accident of this example; Lecture 11 explains how many switches a linear
system can need.

## 6. Pointing: Zermelo's boat

The fourth control set is a direction. In 1931 Ernst Zermelo, best known for
the axioms of set theory (and for an early theorem on chess), posed the
**navigation problem**: a boat moves with fixed speed $V$ relative to the
water, through a current that varies with position; which heading gets it to
a destination soonest? It is often said to have been prompted by the airship
*Graf Zeppelin*'s flight around the world in 1929, a voyage through winds of
exactly this kind.

With a current $k y$ along $x$, the dynamics are $\dot x = V u_x + k y$,
$\dot y = V u_y$ with $(u_x, u_y)$ a unit vector, and
$H = 1 + \lambda_x (V u_x + k y) + \lambda_y V u_y$. Minimizing over unit
vectors points the boat against $(\lambda_x, \lambda_y)$. Since $x$ is cyclic,
$\lambda_x$ is constant, while $\dot\lambda_y = -k\lambda_x$; the heading
angle $\theta$ then obeys Zermelo's law $\tan\theta = \tan\theta_0 - k t$.

In [ ]:
V, k, xT, yT = 1.0, 1.0, 1.0, 1.0
zr = Homotopy(examples.zermelo()).solve([V, k, xT, yT], ([-2, -2], [2, 2]), final_time=(0.2, 5.0))
trz = zr.shooting.trajectory(zr.solution.v, n_points=201)
theta = np.arctan2(trz["u"][:, 1], trz["u"][:, 0])
drift = np.ptp(np.tan(theta) + k * trz["t"])
print(f"t_f = {zr.solution.tf:.6f};  tan(theta) + k t varies by {drift:.1e} along the extremal")

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(trz["z"][:, 0], trz["z"][:, 1], color=BLUE, lw=2, label="extremal")
for i in range(0, len(trz["t"]), 25):
    ax.arrow(trz["z"][i, 0], trz["z"][i, 1], 0.12 * trz["u"][i, 0], 0.12 * trz["u"][i, 1],
             head_width=0.03, color=ORANGE)
yy = np.linspace(0, 1.2, 7)
ax.quiver(np.full_like(yy, -0.15), yy, k * yy, 0 * yy, color=GRAY, scale=6, width=0.004)
ax.set(xlabel="x", ylabel="y", title="heading (orange) and current (gray)")
ax.set_aspect("equal")
ax.grid(alpha=0.3)
fig.tight_layout()

The heading turns steadily clockwise, at exactly the rate Zermelo's law
dictates: the boat climbs steeply while the current is weak, then leans
downstream to ride the faster water near the top, since the target lies
downstream anyway. The law holds to round-off along the computed extremal.

The same law, with $\lambda$ restricted to the velocity costates, points a
rocket's thrust in spacecraft trajectory optimization, where Derek Lawden
called $-\lambda_v$ the **primer vector** (*Optimal Trajectories for Space
Navigation*, 1963). Lecture 16 returns to it, with the
throttle as a second, bang-bang control, as in the package's
`planar_low_thrust_fuel` example:

In [ ]:
lt = derive(examples.planar_low_thrust_fuel())
for u, law in lt.control_law.items():
    print(f"{u} = {law}")
print("throttle switching function:", sp.simplify(lt.switching_functions[0].expr))

## Summary

- The minimum principle replaces $H_u = 0$ with "minimize $H$ over the
  admissible controls, at every instant". Needle variations explain why; the
  canonical equations and boundary conditions do not change.
- For the control sets in practice the minimization has closed forms: the
  stationary point, its clip to a box, the bang-bang law on the sign of the
  switching function, and the unit vector against $b$.
- Bounds change the costate, not just the control: clipping an unbounded
  solution is not optimal.
- `derive` picks the law from the form of $H$ and records each bang-bang
  control's switching function and bounds in `switching_functions`.

## Exercises

1. Give the double integrator of section 4 the bound $|u| \le 1.4$, then
   $1.05$. How does the interior ramp change? Show that the move is possible
   at all in $T = 2$ only for $u_{\max} \ge 4a/T^2 = 1$, and say what the
   optimal control becomes at exactly that bound.
2. Add $\varepsilon u^2/2$ to the minimum-time cost of section 5. Derive the
   law by hand and check `derive`. What happens to it as $\varepsilon \to 0$?
   (Lecture 10 uses exactly this.)
3. In Zermelo's problem, set the current to $k y^2$. Is $\lambda_x$ still
   constant? Does Zermelo's law survive in some form?

## Further reading

- L. S. Pontryagin, V. G. Boltyanskii, R. V. Gamkrelidze and
  E. F. Mishchenko, *The Mathematical Theory of Optimal Processes* (1962),
  chapter 1.
- R. V. Gamkrelidze, "Discovery of the maximum principle", *Journal of
  Dynamical and Control Systems* 5(4), 1999: the story told by one of the
  four.
- D. Liberzon, *Calculus of Variations and Optimal Control Theory* (2012),
  chapter 4: a careful proof with needle variations.
- H. J. Sussmann and J. C. Willems, "300 years of optimal control" (1997), on
  Hestenes and the RAND memorandum.